<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-08-tool-use-and-mcp/lesson-8.7-production-mcp-patterns/notebooks/exercises-8.7.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8.7 — Production MCP Patterns
Netsetos GenAI Course v2.0 | Module 8: MCP & Tool Orchestration

Rate limiting (token bucket, per-tool), circuit breaker + retry + backoff, structured JSON logging, production server template, load testing.

8 exercises. Pure Python — no API keys needed. Run top-to-bottom.

In [ ]:
import time, random, json, logging, functools
from collections import defaultdict
from datetime import datetime, timezone
from enum import Enum
print('Module 8: MCP & Tool Orchestration')
print('Lesson 8.7: Production MCP Patterns')

---
## Ex 1: Token Bucket Rate Limiter
Sliding window — no boundary-burst exploit.

In [ ]:
class RateLimiter:
    def __init__(self, max_calls=60, window_seconds=60):
        self.max_calls = max_calls
        self.window = window_seconds
        self.clients = defaultdict(list)
    def check(self, client_id='default'):
        now = time.time()
        cutoff = now - self.window
        self.clients[client_id] = [t for t in self.clients[client_id] if t > cutoff]
        if len(self.clients[client_id]) >= self.max_calls:
            oldest = min(self.clients[client_id])
            return {'allowed': False, 'remaining': 0,
                    'retry_after_seconds': round(oldest + self.window - now, 1)}
        self.clients[client_id].append(now)
        return {'allowed': True, 'remaining': self.max_calls - len(self.clients[client_id])}

rl = RateLimiter(max_calls=5, window_seconds=60)
for i in range(7):
    r = rl.check('client-001')
    flag = 'OK ' if r['allowed'] else 'BLK'
    print(f'  Call {i+1}: {flag} | {r}')

---
## Ex 2: Per-Tool Rate Limits
Tier limits by tool impact.

In [ ]:
class ToolRateLimiter:
    def __init__(self, limits=None):
        self.limiters = {tool: RateLimiter(max_calls=n) for tool, n in (limits or {}).items()}
        self.default = RateLimiter(max_calls=60)
    def check(self, tool, client_id='default'):
        return self.limiters.get(tool, self.default).check(client_id)

trl = ToolRateLimiter({'search_courses': 30, 'send_email': 5, 'create_order': 10})
print('Per-tool rate limit demo:')
for tool, n in [('search_courses', 6), ('send_email', 6)]:
    print(f'\n  Tool: {tool} (limit varies)')
    for i in range(n):
        r = trl.check(tool, 'client-001')
        flag = 'OK ' if r['allowed'] else 'BLK'
        print(f'    Call {i+1}: {flag} | remaining={r.get("remaining", 0)}')

---
## Ex 3: Circuit Breaker State Machine
CLOSED → OPEN → HALF_OPEN → CLOSED.

In [ ]:
class CircuitState(Enum):
    CLOSED = 'closed'
    OPEN = 'open'
    HALF_OPEN = 'half_open'

class CircuitBreaker:
    def __init__(self, failure_threshold=3, recovery_timeout=2):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failures = 0
        self.state = CircuitState.CLOSED
        self.last_failure_time = 0
    def can_proceed(self):
        if self.state == CircuitState.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = CircuitState.HALF_OPEN
                return True
            return False
        return True
    def record_success(self):
        self.failures = 0
        self.state = CircuitState.CLOSED
    def record_failure(self):
        self.failures += 1
        self.last_failure_time = time.time()
        if self.failures >= self.failure_threshold:
            self.state = CircuitState.OPEN

cb = CircuitBreaker(failure_threshold=3, recovery_timeout=2)
print(f'Initial state: {cb.state.value}')
for _ in range(3):
    cb.record_failure()
print(f'After 3 failures: state={cb.state.value}, can_proceed={cb.can_proceed()}')
time.sleep(2.1)
print(f'After timeout: can_proceed={cb.can_proceed()} -> state={cb.state.value}')
cb.record_success()
print(f'After test success: state={cb.state.value}')

---
## Ex 4: Exponential Backoff with Jitter
Spread retries — no thundering herd.

In [ ]:
def backoff_delay(attempt, base=0.5, max_delay=8):
    delay = min(base * (2 ** attempt), max_delay)
    jitter = delay * random.uniform(0, 0.3)
    return delay + jitter

print('Backoff schedule (with jitter):')
for a in range(6):
    samples = [round(backoff_delay(a), 2) for _ in range(3)]
    print(f'  attempt {a}: samples={samples}')

print('\n1000-client spread at attempt=2 (base=0.5, jitter up to 30%):')
delays = sorted(backoff_delay(2) for _ in range(1000))
print(f'  min={delays[0]:.3f}s  median={delays[500]:.3f}s  max={delays[-1]:.3f}s')
print(f'  -> spread across {(delays[-1] - delays[0]):.2f}s window (vs 0s without jitter)')

---
## Ex 5: Resilient Tool Wrapper
Circuit breaker + retry + backoff in one wrapper.

In [ ]:
class ResilientToolWrapper:
    def __init__(self, handler, max_retries=3, base_delay=0.05, max_delay=0.5):
        self.handler = handler
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.max_delay = max_delay
        self.circuit = CircuitBreaker(failure_threshold=5, recovery_timeout=1)
    def call(self, **kwargs):
        if not self.circuit.can_proceed():
            return {'success': False, 'error': 'circuit_open'}
        for attempt in range(self.max_retries):
            try:
                result = self.handler(**kwargs)
                self.circuit.record_success()
                return {'success': True, 'result': result, 'attempts': attempt + 1}
            except Exception as e:
                self.circuit.record_failure()
                if attempt < self.max_retries - 1:
                    delay = min(self.base_delay * (2 ** attempt), self.max_delay)
                    time.sleep(delay + delay * random.uniform(0, 0.3))
                else:
                    return {'success': False, 'error': str(e), 'attempts': attempt + 1,
                            'circuit_state': self.circuit.state.value}

random.seed(7)
def flaky(query):
    if random.random() < 0.6:
        raise ConnectionError('upstream timeout')
    return f'result for {query}'

wrapped = ResilientToolWrapper(flaky)
print('20 calls to a 60%-failing API:')
successes = 0
for i in range(20):
    r = wrapped.call(query=f'q{i}')
    if r['success']:
        successes += 1
    print(f'  call {i+1:2d}: success={r["success"]}, attempts={r.get("attempts", 0)}, state={wrapped.circuit.state.value}')
print(f'\nTotal successes: {successes}/20  (circuit ended in: {wrapped.circuit.state.value})')

---
## Ex 6: Structured JSON Logger
One JSON line per tool call — Cloud Logging / Datadog parse for free.

In [ ]:
class MCPLogger:
    def __init__(self, service_name='mcp-server'):
        self.service = service_name
    def log_tool_call(self, tool, args, result, duration_ms, client_id='default'):
        entry = {
            'severity': 'ERROR' if not result.get('success', True) else 'INFO',
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'service': self.service, 'type': 'tool_call', 'tool': tool,
            'client_id': client_id,
            'args': {k: str(v)[:100] for k, v in args.items()},
            'success': result.get('success', True),
            'duration_ms': round(duration_ms, 1),
            'result_size': len(json.dumps(result, default=str)) if result else 0,
        }
        if not result.get('success', True):
            entry['error'] = str(result.get('error', ''))[:200]
        print(json.dumps(entry))
        return entry
    def instrument(self, tool_name):
        def deco(func):
            @functools.wraps(func)
            def wrapper(**kwargs):
                start = time.time()
                try:
                    result = func(**kwargs)
                    self.log_tool_call(tool_name, kwargs, {'success': True}, (time.time() - start) * 1000)
                    return result
                except Exception as e:
                    self.log_tool_call(tool_name, kwargs, {'success': False, 'error': str(e)}, (time.time() - start) * 1000)
                    raise
            return wrapper
        return deco

logger = MCPLogger('netsetos-mcp')

@logger.instrument('search_courses')
def search_courses(query, max_price=50000):
    time.sleep(0.05)
    return [{'name': 'GenAI Complete', 'price': 29999}]

print('Structured JSON logs (one per call):')
search_courses(query='AI', max_price=35000)
search_courses(query='ML', max_price=20000)

---
## Ex 7: Production Server Template (assembled)
All patterns wired together — request lifecycle traced.

In [ ]:
trl = ToolRateLimiter({'search': 30, 'send_email': 5})
logger = MCPLogger('production')

def handle_tool(tool, client_id, fn, kwargs):
    start = time.time()
    rl = trl.check(tool, client_id)
    if not rl['allowed']:
        logger.log_tool_call(tool, kwargs, {'success': False, 'error': 'rate_limited'},
                              (time.time() - start) * 1000, client_id)
        return {'error': 'rate_limited', 'retry_after_seconds': rl['retry_after_seconds']}
    try:
        result = fn(**kwargs)
        logger.log_tool_call(tool, kwargs, {'success': True},
                              (time.time() - start) * 1000, client_id)
        return {'success': True, 'result': result}
    except Exception as e:
        logger.log_tool_call(tool, kwargs, {'success': False, 'error': str(e)},
                              (time.time() - start) * 1000, client_id)
        return {'success': False, 'error': str(e)}

def real_search(q):
    return [{'name': 'GenAI Complete', 'price': 29999}]

print('--- Request lifecycle: 7 calls to send_email (limit 5/min) ---')
for i in range(7):
    r = handle_tool('send_email', 'client-001', real_search, {'q': f'msg-{i}'})
    # logs printed by logger; r is the response payload

---
## Ex 8: Load Test Simulation
Simulate 200 mixed requests, observe pattern signatures in logs.

In [ ]:
trl2 = ToolRateLimiter({'search': 50, 'send_email': 10})
stats = {'allowed': 0, 'rate_limited': 0, 'errors': 0}
random.seed(42)

for i in range(200):
    tool = random.choice(['search', 'search', 'send_email'])  # search is more common
    client = f'client-{random.randint(1, 5)}'
    rl = trl2.check(tool, client)
    if not rl['allowed']:
        stats['rate_limited'] += 1
    else:
        stats['allowed'] += 1

print('Load test summary (200 calls across 5 clients):')
print(f'  allowed:      {stats["allowed"]}')
print(f'  rate_limited: {stats["rate_limited"]}')
print(f'  errors:       {stats["errors"]}')
print(f'\nProduction goal: rate_limited count > 0 means your limits are doing work.')
print('In Cloud Logging, filter: `severity=ERROR OR jsonPayload.error="rate_limited"`')

---
## Recap

**4 patterns + 1 health endpoint** = a production-ready MCP server.

1. **Rate limiting** — sliding-window token bucket, per-client × per-tool, retry_after in response.
2. **Circuit breaker** — CLOSED/OPEN/HALF_OPEN; protects upstream from being hammered.
3. **Retry with jittered backoff** — recovers from transient failures without thundering-herd risk.
4. **Structured JSON logging** — one line per call, parseable by Cloud Logging / Datadog / Loki.
5. **Health endpoint** — exposes per-tool circuit state for autoscaler and on-call.

**Order matters:** rate-check first (cheap reject) → circuit-check → retry-loop → log → return.

**Module 8 complete.** Next: Module 9 — AI Agents (LangGraph, multi-agent, memory, computer use, HITL).